# eCREAM Analysis 1: Who is seen by a doctor sooner, dyspnoea or TLoC?

_The main analysis of the live demo. Both synthetic datasets. About ten minutes._

**Before you start.** The eCREAM federation holds the UC1 data model (265 variables in four
population blocks) and, today, two **synthetic** datasets of 5,000 subjects, one with dyspnoea
(`DYSP5K`) and one with transient loss of consciousness, TLoC (`TLOC5K`), generated in July 2026
from anonymized mockup samples. No real patient data are involved. Two properties of the
synthetic data shape every result: each cohort was generated from its own sample, so contrasts
*between* the datasets survive; each block of variables was resampled on its own, so
associations *across* blocks (age and death, saturation and death) do not. Known artefacts:
uniform triage codes and modes of arrival, door-to-nurse at zero, no "discharged" exit, ages
above 89 not capped.

**How this notebook works.** Every analysis is a *federated* call: the code selects the datasets
and the variables, defines a cohort with filters and asks the MIP for aggregates (counts, means,
a test, a model). No record ever reaches this notebook. Run the cells in order; each step says
what to look for and what result to expect. The same steps, for the web interface, are in
`docs/teaching_scenarios/`.

**What this analysis shows.** The time from arrival to the first contact with a doctor is a
quality-of-care indicator that depends on how the department runs, not on the disease: a
patient struggling to breathe is visibly unwell and gets a doctor fast; a patient who fainted
and has since recovered looks fine in the waiting room and waits. The UC1 data model records
this delay in minutes (`door_doctor_delay`, Door to doctor), derived from the timestamps before
anonymization: the federation keeps the clock without keeping the dates. The question: **are
patients with dyspnoea seen sooner than patients with TLoC?** One variable, a clinically obvious
direction, and a real contrast on the synthetic data because each cohort was generated from its
own sample.

## 1. Connect to the eCREAM federation and open the UC1 pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("UC1")         # add version="eCREAMv8" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
def dataset(code, label):
    # by the code, which is how the federation labels its datasets (seen 2026-10-12), else by the long label
    try:
        return dm.datasets[code]
    except (KeyError, LookupError):
        return dm.datasets[label]

dysp = dataset("DYSP5K", "Synthetic Dyspnea dataset (5000 subjects)")
tloc = dataset("TLOC5K", "Synthetic TLoC dataset (5000 subjects)")
DATASETS = [dysp, tloc]                   # every call of this notebook runs on both synthetic datasets
display(dm)
display(dm.datasets.to_frame())           # 21 dataset codes: four hold data today, 17 are the EDs' containers

In [ ]:
# The variable tree: the four population blocks of the UC1 case report form. The demo uses
# ALL_PATIENTS (logistics, demographics, door-to-event delays, dataset of origin) and, for the
# mortality model, two variables of DYSPNEA_AND_TLOC (vital signs, follow-up).
dm.tree(group="ALL_PATIENTS", include_variables=True)

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the analysis; come back to it
# when you want to see how a result payload is unpacked.
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the selected datasets
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False, datasets=None):
    """A Pipeline on the datasets of this notebook (DATASETS, or the ones given) with the given
    variables and cohort filter. Tests and models need drop_missing=True: the platform requires an
    explicit missing-values step and analyses complete cases only. describe() keeps it off so it
    can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=datasets or DATASETS, variables=variables),
                        filters=filters, handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def name_of(dataset):
    """Column name of a dataset in the tables: its code, or 'all datasets' for the pooled row."""
    return POOLED if dataset == POOLED else dataset._code


def rows_for(result, variable, dataset=POOLED):
    """The describe row of one variable: the pooled row by default, or the row of one dataset."""
    keys = (POOLED,) if dataset == POOLED else (dataset._code, dataset.label)
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") in keys]


def numeric_by_dataset(result, variable, datasets=None):
    """n, missing, mean, SD, min, quartiles and max of a numerical variable, one row per dataset
    and the pooled row, from a describe result."""
    rows = []
    for ds in list(datasets or DATASETS) + [POOLED]:
        for r in rows_for(result, variable, ds):
            d = r.get("data") or {}
            rows.append({"dataset": name_of(ds), "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "min": d.get("min"), "q1": d.get("q1"),
                         "median": d.get("q2"), "q3": d.get("q3"), "max": d.get("max")})
    return pd.DataFrame(rows)


def counts_by_dataset(result, variable, datasets=None):
    """Counts and percentages of the levels of a categorical variable, one column pair per dataset
    and for the pooled row; the number of missing values in .attrs['missing']."""
    lab = labels_of(variable)
    table, missing = {}, {}
    for ds in list(datasets or DATASETS) + [POOLED]:
        for r in rows_for(result, variable, ds):
            d = r.get("data") or {}
            counts = {str(k): v for k, v in (d.get("counts") or {}).items()}
            total = sum(counts.values())
            table[name_of(ds)] = pd.Series({label: counts.get(c, 0) for c, label in lab.items()})
            table[name_of(ds) + " %"] = pd.Series({label: 100 * counts.get(c, 0) / total if total else np.nan
                                                   for c, label in lab.items()})
            missing[name_of(ds)] = d.get("num_na")
    out = pd.DataFrame(table)
    out.index.name = variable.label
    out.attrs["missing"] = missing
    return out


def record_counts(filters, variable, datasets=None):
    """Records matching a filter, per dataset and pooled, from a describe of one variable that is
    never missing (the dataset of origin): the notebook's version of the filter counter."""
    res = analysis([variable], filters, datasets=datasets).describe()
    out = {}
    for ds in list(datasets or DATASETS) + [POOLED]:
        rows = rows_for(res, variable, ds)
        d = (rows[0].get("data") or {}) if rows else {}
        out[name_of(ds)] = d.get("num_total")          # None when fewer than 10 records match
    return out


def share_of(variable, level, filters, datasets=None):
    """% of the records with a given level (code) of a categorical variable among the records
    where it is recorded, pooled over the datasets, for a cohort filter."""
    res = analysis([variable], filters, datasets=datasets).describe()
    rows = rows_for(res, variable)
    counts = ((rows[0].get("data") or {}).get("counts") or {}) if rows else {}
    total = sum(counts.values())
    return (100 * counts.get(level, 0) / total if total else np.nan), total


def histograms_by_dataset(variable, bins=20, filters=None, datasets=None, unit=""):
    """One histogram per dataset (one federated call each), drawn side by side. Bins with fewer
    than 10 records come back empty: the platform's privacy rule."""
    datasets = list(datasets or DATASETS)
    fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 3.4))
    for ax, ds in zip(np.atleast_1d(axes), datasets):
        h = analysis([variable], filters, drop_missing=True, datasets=[ds]).histogram(variable=variable, bins=bins).raw["histogram"][0]
        edges, counts = h["bins"], h["counts"]
        bars = [c or 0 for c in counts]
        ax.bar(edges[:-1], bars, width=np.diff(edges), align="edge", color="#4C72B0")
        hidden = sum(c is None for c in counts)
        peak = int(np.argmax(bars))
        ax.set_title(f"{variable.label}, {ds.label}")
        ax.set_xlabel(f"{variable.label} ({unit})" if unit else variable.label)
        ax.set_ylabel("records")
        print(f"{ds._code}: bins {edges[1] - edges[0]:.1f} wide, peak {bars[peak]} records at "
              f"{edges[peak]:.0f} to {edges[peak + 1]:.0f}, {hidden} bins under 10 records not drawn")
    plt.tight_layout()
    plt.show()


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def show_ttest(result, what, unit=""):
    # one line with the t-test result; a key the platform does not return is shown as n/a
    s = ttest_summary(result)
    num = lambda v, spec: format(v, spec) if isinstance(v, (int, float)) else "n/a"
    print(f"{what}: difference of means {num(s['mean_diff'], '.1f')} {unit} (95% CI {num(s['ci_lower'], '.1f')} "
          f"to {num(s['ci_upper'], '.1f')}), t = {num(s['t_stat'], '.1f')} on {num(s['df'], '.0f')} df, "
          f"p = {fmt_p(s['p'])} ({num(s['p'], '.1e')}), Cohen's d {num(s['cohens_d'], '.2f')}, n = {num(s['n_obs'], ',')}")
    missing = [k for k, v in s.items() if v is None]
    if missing:
        print(f"   (keys the platform did not return: {', '.join(missing)}; keys returned: {', '.join(result.raw)})")
    return s


def chi2_summary(result, x, y):
    """Chi-squared Test result -> chi2, p, dof, and the expected counts as a table with labels."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    lab_x, lab_y = labels_of(x), labels_of(y)
    expected = pd.DataFrame(raw.get("expected") or [],
                            index=[lab_x.get(str(i), str(i)) for i in raw.get("x_labels") or []],
                            columns=[lab_y.get(str(c), str(c)) for c in raw.get("y_labels") or []])
    expected.index.name, expected.columns.name = x.label, y.label
    return raw.get("chi2"), raw.get("p_value", raw.get("p")), raw.get("dof"), expected


def term_label(term, variables):
    """Model term -> readable label: 'sex[2]' -> 'Sex = Female', 'age' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with coefficient, odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(lambda t: term_label(t, variables))
    for col in ("coefficient", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "coefficient", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"coefficient": "{:.4f}", "odds_ratio": "{:.3f}", "ci_low": "{:.3f}",
                                "ci_high": "{:.3f}", "p": "{:.3f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 0.6 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def roc_and_matrix(cv, event="death", non_event="survivor"):
    """Cross-validation result -> the ROC curve of every fold with its area, and the confusion
    matrix summed over the folds at the platform's 50% cut-off."""
    curves, cm = cv.raw["roc_curves"], cv.raw["confusion_matrix"]
    aucs = [c["auc"] for c in curves]
    print("Area under the ROC curve, per fold:", ", ".join(f"{a:.2f}" for a in aucs), f"| average {np.mean(aucs):.2f}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for c in curves:
        axes[0].plot(c["fpr"], c["tpr"], linewidth=1, label=f"{c['name'].replace('_', ' ')}: area {c['auc']:.2f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey", label="coin toss: area 0.50")
    axes[0].set_xlabel(f"share of {non_event}s wrongly called {event}s")
    axes[0].set_ylabel(f"share of {event}s caught")
    axes[0].set_title("ROC curve, on patients the model had not seen")
    axes[0].legend(fontsize=8, loc="lower right")
    matrix = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])  # rows: the truth; columns: the verdict
    axes[1].imshow(matrix, cmap="Blues")
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, f"{matrix[i, j]:,}", ha="center", va="center", fontsize=14,
                         color="white" if matrix[i, j] > matrix.max() / 2 else "black")
    axes[1].set_xticks([0, 1])
    axes[1].set_xticklabels([f"called a {non_event}", f"called a {event}"])
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels([f"was a {non_event}", f"was a {event}"])
    axes[1].set_title("Confusion matrix at the platform's 50% cut-off")
    plt.tight_layout()
    plt.show()
    n = sum(cm.values())
    sens = 100 * cm["tp"] / (cm["tp"] + cm["fn"]) if cm["tp"] + cm["fn"] else float("nan")
    spec = 100 * cm["tn"] / (cm["tn"] + cm["fp"]) if cm["tn"] + cm["fp"] else float("nan")
    print(f"Accuracy {100 * (cm['tp'] + cm['tn']) / n:.1f}%, sensitivity {sens:.1f}%, specificity {spec:.1f}%")
    return aucs


def saved_experiments():
    """The experiments saved from the web interface (My experiments), as the platform lists them."""
    items = client.experiments().list()
    if not items:
        print("No saved experiment is listed (nothing saved yet, or the registry answers differently on this platform).")
        return pd.DataFrame()
    return pd.DataFrame(items)


print("Toolbox ready.")

## 3. The datasets and the variables of this analysis

| Code | Label in the MIP | Tree location (under UC1) | Role | Values |
|---|---|---|---|---|
| `dataset` | Dataset Registry origin | ALL_PATIENTS / Dataset Registry Origin | which cohort a record belongs to; grouping variable of the t-test | 21 codes, listed in the tree |
| `door_doctor_delay` | Door to doctor | ALL_PATIENTS / TimeMetrics / Process Milestones | outcome: minutes from arrival to the doctor | integer, minutes |
| `sex` | Sex | ALL_PATIENTS / Demographics | described alongside, as in the demo | 1 Male; 2 Female; 3 Unknown |
| `age` | Age | ALL_PATIENTS / Demographics | described alongside, as in the demo | integer |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `door_doctor_delay`)
is what the csv column is called; you will see it in some result payloads. Both synthetic
datasets are selected for every call.

In [ ]:
delay      = dm.variables["Door to doctor"]
sex        = dm.variables["Sex"]
age        = dm.variables["Age"]
dataset    = dm.variables["Dataset Registry origin"]      # which cohort a record belongs to
display(mip.to_frame([delay, sex, age, dataset]))

## Step 1 and 2. Choose the data and review it before modelling

One `describe()` of the delay, sex and age on both datasets: the platform answers one row per
dataset and one pooled row ("all datasets"), with counts, missing values and the distribution.
Then one histogram per dataset.

*Expected, Door to doctor in minutes:* 5,000 records and no missing value in each dataset;
dyspnoea mean 47.8 (SD 33.4), min 0, quartiles 19, 41, 72, max 193; TLoC mean 105.3 (SD
113.5), min 4, quartiles 30, 53, 145, max 679; all datasets mean 76.5, median 47. The
dyspnoea histogram is compact (bins about 10 minutes wide, peak at 10 to 19 minutes with 1,158
patients); the TLoC histogram peaks in its first bin (1,790 patients under 38 minutes) and has a
long tail; in each the last four bins hold fewer than 10 patients and are not drawn.

In [ ]:
t0 = time.perf_counter()
review = analysis([delay, sex, age]).describe()
print(f"the federation answered in {time.perf_counter() - t0:.1f} s")
display(numeric_by_dataset(review, delay).style.format({c: "{:.1f}" for c in ("mean", "sd")}).hide(axis="index"))
counts = record_counts(None, dataset)
print("records per dataset:", counts)
histograms_by_dataset(delay, bins=20, unit="minutes")

*What it means:* the same variable, two shapes. The medians are 41 and 53 minutes, a difference
of 12 minutes; the means are 47.8 and 105.3, a difference of 57, because one TLoC patient in
seven waits more than four hours. The mean difference the t-test will report is driven by that
tail.

## Step 3. Compare the cohorts: the Independent t-test

Outcome the delay, grouping variable the dataset of origin, group A dyspnoea, group B TLoC.

*Expected:* t = -34.3 on 9,998 degrees of freedom, p of the order of 10^-244; mean difference
-57.4 minutes (95% CI -60.7 to -54.2), Cohen's d -0.69. The sign follows the order of the
groups (dyspnoea minus TLoC). The platform pools the variances; with 5,000 patients on each side
Welch's version gives the same t.

In [ ]:
t0 = time.perf_counter()
t = analysis([delay, dataset], drop_missing=True).t_test(variable=delay, group_by=dataset, group_a="DYSP5K", group_b="TLOC5K")
print(f"the federation answered in {time.perf_counter() - t0:.1f} s")
s = show_ttest(t, "Door to doctor, dyspnoea minus TLoC", "minutes")

## Step 4. Counts at a threshold

"How many wait more than two hours?" The notebook's version of the filter counter: a describe of
the dataset variable under a filter, per dataset.

*Expected:* seen within 30 minutes 41.7% of the dyspnoea patients and 25.0% of the TLoC
patients; within 60 minutes 64.5% and 53.2%; more than 120 minutes 2.4% and 28.3%; more than 240
minutes 0.0% and 14.1%; 180 dyspnoea records carry a delay of exactly 0 minutes. A count under 10
comes back empty (shown as NaN): the platform's privacy rule.

In [ ]:
total = record_counts(None, dataset)
rows = []
for label, f in (("under 30 minutes", F(delay) < 30), ("under 60 minutes", F(delay) < 60),
                 ("over 120 minutes", F(delay) > 120), ("over 240 minutes", F(delay) > 240), ("exactly 0 minutes", F(delay) == 0)):
    n = record_counts(f, dataset)
    rows.append({"delay": label, **{k: n[k] for k in n}, **{f"{k} %": (100 * n[k] / total[k] if n[k] is not None else np.nan) for k in n}})
thresholds = pd.DataFrame(rows)
display(thresholds.style.format({c: PCT for c in thresholds.columns if c.endswith("%")}).hide(axis="index"))

## Step 5. State the result

A model answer:

> On the two synthetic eCREAM cohorts, patients with dyspnoea were seen by a doctor after 48
> minutes on average (median 41) and patients with TLoC after 105 minutes (median 53); the
> difference of means, 57 minutes, is significant (t = -34, p < 0.001) and largely due to the
> long waits of one TLoC patient in seven. Synthetic data, real workflow: the result was
> computed on the federation without any record leaving its dataset.

In the web interface the demo ends with **Save experiment**; the notebook can list what was
saved there.

In [ ]:
display(saved_experiments())

## Extension. The delay by sex, within each cohort

The same t-test with sex as the grouping variable, one dataset at a time.

*Expected:* dyspnoea, men 48.7 against women 46.0 minutes, difference 2.7 (t = 2.7, p = 0.007,
n = 4,983); TLoC 105.1 against 105.5 (t = -0.2, p = 0.88, n = 4,988). Three minutes is nothing
clinically, and in synthetic data whose blocks were resampled independently (sex in
Demographics, the delay in TimeMetrics) a small p-value across blocks is chance, not signal:
the first glimpse of what Analysis 3 shows in full.

In [ ]:
for ds in DATASETS:
    by_sex = analysis([delay, sex], drop_missing=True, datasets=[ds]).t_test(variable=delay, group_by=sex, group_a="1", group_b="2")
    show_ttest(by_sex, f"{ds.label}: men minus women", "minutes")